In [3]:
from data import *

import numpy as np

In [6]:
# generate data
d1 = 1000
d2 = 100
r = 1
p = 5.0 / d

M = get_random_matrix(d1, d2, r) / np.sqrt(d1)
observed_M, masks = get_uniformly_random_samples(M, p)

In [7]:
#eps = 0.01

cov_observe_M =  observed_M.T @ observed_M
cov_observe_count = (observed_M == 0).T @ (observed_M == 0)
diag_cov = np.diag( np.diag(cov_observe_M) )
T = (1.0 / p) * diag_cov + (1.0 / (p**2)) * (cov_observe_M - diag_cov)

np.count_nonzero(cov_observe_M)

9280

In [8]:
cov_observe_count = (1 * (observed_M != 0)).T @ (1 * (observed_M != 0))
cov_observe_count = cov_observe_count + (cov_observe_count == 0) * 1
T = cov_observe_M / (cov_observe_count/d1)

In [9]:
T_masks = np.nonzero(T)
MTM = M.T @ M

mask_err = T[T_masks] - MTM[T_masks]

print(np.linalg.norm(mask_err) / np.linalg.norm(MTM, 'fro'))

0.06662029631520125


In [10]:
# impute missing values from rank-r SVD corresponding to masks

epochs = 100
lr = 0.05
X = T
T_masks = 1 * (T != 0)
for i in range(epochs):
    U, D, Vt = np.linalg.svd(X)
    D[r:] = 0
    X_update = U @ np.diag(D) @ Vt

    X = X * T_masks + X_update * (1 - T_masks)

    err = M.T @ M - X
    relative_err = np.linalg.norm(err, 'fro') / np.linalg.norm(M.T @ M, 'fro') 
    print(relative_err)

0.07378476212562296
0.06706698111733722
0.06692305532250023
0.06691991189916496
0.06691995465960775
0.06691999036205677
0.06691999947324254
0.06692000148144155
0.06692000189798405
0.06692000198136724
0.0669200019976713
0.06692000200080712
0.06692000200140304
0.06692000200151528
0.06692000200153628
0.06692000200154018
0.06692000200154088
0.06692000200154102
0.06692000200154104
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0.06692000200154105
0

In [12]:
# run soft impute
epochs = 1
lr = 0.05
X = T
for i in range(epochs):
    U, D, Vt = np.linalg.svd(X)
    D[r:] = 0
    X_update = U @ np.diag(D) @ Vt

    #if np.linalg.norm(X - X_update, 'fro') / np.linalg.norm(X) < eps:
    #    print(i)
    #    break

    X = (1 - lr) * X + lr * X_update

    # print distance
    err = M.T @ M - X
    relative_err = np.linalg.norm(err, 'fro') / np.linalg.norm(M.T @ M)
    print(relative_err)

#print(D)

0.264250769026682


In [14]:
err = M.T @ M - T
relative_err = np.linalg.norm(err, 'fro') / np.linalg.norm(M.T @ M)
print(relative_err)

0.27656936618151623
